# A1 · Magazzino frigorifero: dai segnali del PLC alla risposta al cliente (C#)

Stessi numeri nel notebook [PySharp](A1-magazzino-frigorifero-pysharp.ipynb) (facciata `Interop`); un test verifica che coincidano. Kernel JupyterNet su .NET 11.

## Il caso

Un **magazzino frigorifero** a +2 °C custodisce merce di un cliente (una società alimentare). Il PLC dell'impianto invia a uno storico, ogni minuto, la temperatura della cella, lo stato di porta e sbrinamento, la corrente del compressore, la pressione di aspirazione e la vibrazione. Un contatore invia i kWh ogni ora.

**Chi lavora sui dati.** Il *responsabile qualità* deve poter certificare che la catena del freddo non si è rotta (soglia HACCP +4 °C). Il *manutentore* vuole sapere in anticipo se il compressore sta cedendo. Il *responsabile di stabilimento* deve stimare i consumi di energia della prossima settimana. Il *cliente* chiede: **«la mia merce è stata sempre sotto +4 °C?»**

**I dati sono sintetici**, generati da `generate-data.cs` con un seme: servono a verificare i risultati contro la verità nota (`data/verita.json`), cosa che con dati reali non è mai possibile.

| Passo | Domanda | Strumento |
|---|---|---|
| 1 | I dati sono completi? | qualità dei dati |
| 2 | Qual è la temperatura *vera* quando il sensore tace? | filtro di Kalman con `NaN` ([cap. 6](../../../../refs/timeseries/00-formule-e-libreria/06-kalman.md)) |
| 3 | Quanta energia consumeremo? | seasonal naive, Holt-Winters, MASE ([cap. 3](../../../../refs/timeseries/00-formule-e-libreria/03-riferimenti-e-metriche.md), [5](../../../../refs/timeseries/00-formule-e-libreria/05-smoothing.md)) |
| 4 | Il compressore sta cedendo? | GARCH sulla vibrazione ([cap. 8](../../../../refs/timeseries/00-formule-e-libreria/08-garch.md)) |
| 5 | Come si influenzano temperatura, corrente e pressione? | VAR ([cap. 7](../../../../refs/timeseries/00-formule-e-libreria/07-var.md)) |
| 6 | KPI, dashboard e risposta al cliente | tile, grafici, testo con i numeri |

In [ ]:
#load "../../../shared/MathemaSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;
using System.IO;
using System.Runtime.CompilerServices;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });
static string Here([CallerFilePath] string path = "") => Path.GetDirectoryName(path)!;   // cartella di questo notebook
static double Num(string s) => s.Length == 0 ? double.NaN : double.Parse(s, CultureInfo.InvariantCulture);
// Valore numerico di una chiave in un JSON piatto (verita.json): bastano due righe, nessuna dipendenza.
static double JsonNumber(string json, string key)
{
    var quote = '"';                                   // niente barre rovesciate: questo codice passa da una stringa Python
    var i = json.IndexOf(quote + key + quote + ":") + key.Length + 3;
    var j = json.IndexOfAny(new[] { ',', (char)10, (char)13 }, i);
    return double.Parse(json[i..j], CultureInfo.InvariantCulture);
}

var data = Path.Combine(Here(), "data");
var cella = File.ReadLines(Path.Combine(data, "cella_1min.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] Column(int i) => cella.Select(r => Num(r[i])).ToArray();
double[] temp = Column(1), current = Column(5), pressure = Column(6), vib = Column(7);
const double limit = 4.0;   // soglia HACCP, °C

## 1 · Acquisire: i dati sono completi?
Un buco di comunicazione non è un dato a zero: è un dato **assente**. Prima di ogni calcolo si misura quanto manca e dove.

In [ ]:
var gaps = new List<(int Start, int Length)>();
for (var i = 0; i < temp.Length; i++)
{
    if (!double.IsNaN(temp[i])) continue;
    var j = i;
    while (j < temp.Length && double.IsNaN(temp[j])) j++;
    gaps.Add((i, j - i));
    i = j;
}
P("campioni", temp.Length);
P("buchi_n", gaps.Count);
P("mancanti_min", gaps.Sum(g => g.Length));
P("buco_max_min", gaps.Max(g => g.Length));

// Un buco è "sospetto" se la temperatura ai suoi bordi differisce di più di 1 °C: durante il buco è successo qualcosa.
var suspect = gaps.Where(g => g.Start > 0 && g.Start + g.Length < temp.Length
                              && Math.Abs(temp[g.Start + g.Length] - temp[g.Start - 1]) > 1.0).ToArray();
P("buchi_sospetti_min", suspect.Sum(g => g.Length));

## 2 · Capire la temperatura: il filtro di Kalman
Il sensore ha un rumore noto (σ = 0.08 °C, dal datasheet). Il modello *local level* tratta la temperatura vera come uno stato nascosto che si muove di poco a ogni minuto (varianza $q$); i `NaN` saltano la correzione. La varianza $q$ si sceglie massimizzando la **log-verosimiglianza** su una griglia.

In [ ]:
const double sensorVariance = 0.08 * 0.08;
var qGrid = new[] { 0.001, 0.003, 0.01, 0.03, 0.1 };
var logLik = qGrid.Select(q => KalmanFilter.LocalLevel<double>(q, sensorVariance, temp[0], 1.0).Filter(temp).LogLikelihood).ToArray();
P("kalman_loglik_grid", logLik);
var qBest = qGrid[Array.IndexOf(logLik, logLik.Max())];
P("kalman_q", qBest);

var level = KalmanFilter.LocalLevel<double>(qBest, sensorVariance, temp[0], 1.0).Filter(temp).FilteredFirstState;
var trend = KalmanFilter.LocalLinearTrend<double>(qBest, qBest / 10, sensorVariance, temp[0], 0.0, 1.0).Filter(temp).FilteredFirstState;

### Il KPI HACCP: quanti minuti sopra +4 °C?
Quattro risposte allo stesso quesito. **La salita avviene dentro un buco di comunicazione**, quindi nessuno strumento la vede per intero: l'unica risposta onesta è un **intervallo**.

In [ ]:
int Above(double[] x) => x.Count(v => v > limit);
int Runs(double[] x) { var n = 0; for (var i = 0; i < x.Length; i++) if (x[i] > limit && (i == 0 || x[i - 1] <= limit)) n++; return n; }

var observed = temp.Count(v => !double.IsNaN(v) && v > limit);
P("haccp_osservati_min", observed);
P("haccp_kalman_livello_min", Above(level));
P("haccp_kalman_trend_min", Above(trend));
P("haccp_escursioni_kalman", Runs(level));
P("temp_max_filtrata", level.Max());
P("haccp_intervallo_min", new[] { (double)observed, observed + suspect.Sum(g => g.Length) });

// Verità: la conosce solo il generatore dei dati sintetici, non uno storico reale.
var truth = File.ReadAllText(Path.Combine(data, "verita.json"));
var truthMinutes = (int)JsonNumber(truth, "minuti_veri_sopra_soglia");
P("haccp_verita_min", truthMinutes);
P("haccp_verita_nell_intervallo", truthMinutes >= observed && truthMinutes <= observed + suspect.Sum(g => g.Length) ? 1 : 0);

In [ ]:
// L'evento: martedì 15 settembre, dalle 13:50 alle 14:40 (minuto 1440 + 830 … 1440 + 880).
int a = 1440 + 13 * 60 + 50, b = 1440 + 14 * 60 + 40;
Chart.Lines("Temperatura durante l'evento (asse x: minuti dalle 13:50)",
    new Line("misura (buco = vuoto)", temp[a..b]),
    new Line("Kalman livello", level[a..b]),
    new Line("Kalman trend", trend[a..b]),
    new Line("soglia +4 °C", Enumerable.Repeat(limit, b - a).ToArray(), Dashed: true));

## 3 · Prevedere l'energia della prossima settimana
Sei settimane di kWh orari: cinque di addestramento, l'ultima come **prova a cui il modello non ha mai guardato**. Il confronto è con la MASE rispetto al seasonal naive settimanale ($m=168$): sotto 1 il modello batte il riferimento.

In [ ]:
var energia = File.ReadLines(Path.Combine(data, "energia_oraria.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] kwh = energia.Select(r => Num(r[1])).ToArray();
double[] train = kwh[..840], test = kwh[840..];

var candidates = new (string Name, double[] Forecast)[]
{
    ("sn24", ReferenceForecasts.SeasonalNaive<double>(train, 24, 168)),
    ("sn168", ReferenceForecasts.SeasonalNaive<double>(train, 168, 168)),
    ("hw24", ExponentialSmoothing.HoltWinters<double>(train, 24, 0.2, 0.01, 0.2, 168, 1.0).Forecast),
    ("hw168", ExponentialSmoothing.HoltWinters<double>(train, 168, 0.2, 0.01, 0.2, 168, 1.0).Forecast),
};
foreach (var (name, f) in candidates)
{
    P($"mase_{name}", Accuracy.Mase<double>(test, f, train, 168));
    P($"kwh_sett_{name}", f.Sum());
}
P("kwh_sett_reale", test.Sum());
foreach (var (name, f) in candidates) P($"kwh_errore_pct_{name}", 100 * (f.Sum() - test.Sum()) / test.Sum());
var best = candidates.OrderBy(c => Accuracy.Mase<double>(test, c.Forecast, train, 168)).First();
Console.WriteLine($"modello_scelto: {best.Name}");

In [ ]:
Chart.Lines("Ultima settimana: consumo reale e previsioni (kWh/h)",
    new Line("reale", test), new Line("seasonal naive 168", candidates[1].Forecast, Dashed: true), new Line("Holt-Winters 168", candidates[3].Forecast, Dashed: true));

> **Due criteri, due risposte.** Per il **profilo orario** (MASE) vince il seasonal naive settimanale; per il **totale della settimana** (il budget di energia) è Holt-Winters a sbagliare meno. E con **una sola settimana di prova** la scelta del modello e la sua valutazione coincidono: il risultato è ottimista. La validazione seria ripete la prova su molte origini (*rolling origin*, [infografica sulla validazione](../../../../refs/timeseries/)): è fra le teorie del libro non ancora implementate.

## 4 · Il compressore sta cedendo? La volatilità della vibrazione
La vibrazione dipende dallo stato del compressore (acceso/spento). Tolto il livello di ciascuno stato, restano shock la cui **varianza** dice quanto il compressore è "inquieto". Il GARCH si addestra sulle **prime 48 ore** (comportamento normale) e poi si legge la volatilità condizionale sulle 24 ore successive.

In [ ]:
var on = current.Select(c => c > 10).ToArray();
double MeanState(bool state) { double s = 0; var n = 0; for (var i = 0; i < 2880; i++) if (on[i] == state) { s += vib[i]; n++; } return s / n; }
var mOn = MeanState(true); var mOff = MeanState(false);
P("vib_livello_on_off", new[] { mOn, mOff });
var residual = vib.Select((v, i) => v - (on[i] ? mOn : mOff)).ToArray();

var garch = Garch11.FitVarianceTargeting<double>(residual[..2880]);
P("garch_fit", new[] { garch.Omega, garch.Alpha, garch.Beta });
var sigmaBar = Math.Sqrt(garch.UnconditionalVariance);
P("garch_sigma_base", sigmaBar);
var sigma = garch.ConditionalVariances(residual).Select(Math.Sqrt).ToArray();
var alarm = sigma.Select(s => s > 1.5 * sigmaBar).ToArray();
double Rate(int from, int to) { var n = 0; for (var i = from; i < to; i++) if (alarm[i]) n++; return (double)n / (to - from); }
P("allarme_giorno1", Rate(0, 1440));
P("allarme_giorno2", Rate(1440, 2880));
P("allarme_giorno3", Rate(2880, 4320));
// Allarme sostenuto: almeno 8 minuti sopra soglia nell'ultima ora (finestra mobile di 60 minuti).
int firstAlarm = -1, window = 0;
for (var i = 0; i < alarm.Length; i++)
{
    if (alarm[i]) window++;
    if (i >= 60 && alarm[i - 60]) window--;
    if (window >= 8 && firstAlarm < 0) firstAlarm = i;
}
P("allarme_primo_minuto", firstAlarm);
P("allarme_ritardo_min", firstAlarm - 2880);   // il cambiamento vero è al minuto 2880 (48 h): negativo = falso allarme

In [ ]:
Chart.Lines("Volatilità σ_t della vibrazione e soglia (1 punto ogni 15 min)",
    new Line("σ_t", Enumerable.Range(0, sigma.Length / 15).Select(i => sigma[i * 15]).ToArray()),
    new Line("soglia", Enumerable.Repeat(1.5 * sigmaBar, sigma.Length / 15).ToArray(), Dashed: true));

## 5 · Segnali accoppiati: VAR
Medie su 5 minuti di temperatura (filtrata), corrente e pressione. Il VAR(1) risponde: *quanto la corrente di adesso predice la temperatura fra cinque minuti?*

In [ ]:
const int step = 5;
var n5 = temp.Length / step;
var series = new double[n5, 3];
for (var k = 0; k < n5; k++)
{
    double st = 0, sc = 0, sp = 0;
    for (var j = 0; j < step; j++) { st += level[k * step + j]; sc += current[k * step + j]; sp += pressure[k * step + j]; }
    series[k, 0] = st / step; series[k, 1] = sc / step; series[k, 2] = sp / step;
}
var var3 = VarModel.Fit<double>(series, 1);
var a1 = var3.Coefficients[0];
P("var_a1", new[] { a1[0,0], a1[0,1], a1[0,2], a1[1,0], a1[1,1], a1[1,2], a1[2,0], a1[2,1], a1[2,2] });

Lettura di $A_1$ (righe = equazioni, colonne = temperatura, corrente, pressione al passo precedente): **temperatura ← corrente** è negativa (compressore acceso ⇒ la cella si raffredda); **corrente ← temperatura** è fortemente positiva (cella calda ⇒ il termostato accende il compressore). Il VAR ritrova, dai soli dati, la logica del termostato.

## 6 · Comunicare: KPI, dashboard e risposta al cliente

In [ ]:
var minutes72 = temp.Length;
var energyNext = best.Forecast.Sum();
var alarmDay3 = Rate(2880, 4320);

Kpi.Tiles(
    new Tile("Tempo sopra +4 °C", $"{observed}–{observed + suspect.Sum(g => g.Length)} min", "minimo osservato – massimo non escludibile", "warn"),
    new Tile("Escursioni HACCP", $"{Runs(level)}", $"temperatura massima stimata {level.Max():F1} °C", "warn"),
    new Tile("Dati mancanti", $"{100.0 * gaps.Sum(g => g.Length) / minutes72:F1} %", $"{gaps.Count} buchi, il più lungo {gaps.Max(g => g.Length)} min", "ok"),
    new Tile("Energia prossima settimana", $"{energyNext:F0} kWh", $"modello {best.Name}, MASE {Accuracy.Mase<double>(test, best.Forecast, train, 168):F2}; sul totale {100 * (energyNext - test.Sum()) / test.Sum():+0.0;-0.0} % nella settimana di prova", "ok"),
    new Tile("Vibrazione compressore", $"{100 * alarmDay3:F0} % allarme", $"ultime 24 h, contro {100 * Rate(0, 1440):F1} % il primo giorno; primo allarme {firstAlarm - 2880} min dopo il cambio", alarmDay3 > 0.2 ? "alarm" : "ok"));

In [ ]:
var answer = $"Nelle 72 ore dal 14 al 16 settembre la temperatura della cella è stata osservata sopra +4 °C per almeno {observed} minuti, in {Runs(level)} escursione, con un massimo stimato di {level.Max():F1} °C (martedì pomeriggio). " +
             $"In {suspect.Sum(g => g.Length)} minuti il sensore non ha comunicato e la temperatura ai bordi del buco differiva di oltre 1 °C: non possiamo escludere altri minuti sopra soglia, per un massimo di {observed + suspect.Sum(g => g.Length)}. " +
             $"Per questo la nostra risposta non è «sempre sotto +4 °C», ma «da {observed} a {observed + suspect.Sum(g => g.Length)} minuti sopra soglia in un'unica escursione»; proponiamo di verificare la porta della cella e il gateway di comunicazione.";
Kpi.Quote("Risposta al cliente: «la mia merce è stata sempre sotto +4 °C?»", answer);

var maintenance = $"La volatilità della vibrazione del compressore è oltre soglia nel {100 * alarmDay3:F0} % dell'ultimo giorno (contro il {100 * Rate(0, 1440):F1} % del primo). L'allarme sostenuto è scattato {firstAlarm - 2880} minuti dopo l'inizio del cambiamento: consigliato un controllo del cuscinetto.";
Kpi.Quote("Nota per la manutenzione", maintenance);

## Cosa abbiamo imparato
- **Un buco nei dati è un'informazione**: il KPI HACCP si dà come intervallo, e la verità nota (11 minuti) cade dentro.
- **Il Kalman fa il suo lavoro ma non indovina**: non vede una salita avvenuta interamente nel buco, né con il solo livello né con il trend.
- **MASE batte la percentuale**: i kWh non sono mai zero, ma il metodo è lo stesso per la produzione fotovoltaica, che di notte vale zero (scenario D).
- **GARCH come campanello**: non dice *cosa* sta cedendo, dice che il comportamento è cambiato; l'allarme non coincide con il cambiamento (*change time is not alarm time*).
- **Il PySharp dà gli stessi numeri**: stessa libreria, stessa risposta, linguaggio a scelta di chi lavora sull'impianto.